# ADAN Trading Bot - Google Colab GPU Pipeline

**Complete training pipeline for Google Colab (T4 GPU)**

This notebook provides a 5-step pipeline:
1. **Setup** - Clone repository and install dependencies
2. **Data** - Download 50,000 real candles via CCXT
3. **Train** - Run PBT training with 4 worker profiles on T4
4. **Extract** - Extract the best model to production
5. **Paper Trading** - Run isolated virtual paper trading

---

**Architecture:**
- PPO + ContextualTemporalFusionExtractor (FiLM Meta-RL)
- Multi-timeframe: 5m (master clock), 1h, 4h
- Capital Tiers: Micro ($11-$30) -> Enterprise ($1000+)
- Virtual wallet: $20.50 initial, completely isolated


## 0. Verify GPU
Make sure you have a T4 GPU allocated (Runtime > Change runtime type > GPU).


In [ ]:
# Verify GPU availability
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU')


## 1. Setup - Clone Repository & Install Dependencies


In [ ]:
# Clone the ADAN repository
import os

REPO_URL = 'https://github.com/Cabrel10/ADAN0.git'
REPO_DIR = '/content/ADAN0'

if os.path.exists(REPO_DIR):
    print(f'Repository already exists at {REPO_DIR}')
    %cd {REPO_DIR}
    !git pull origin main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

print(f'Working directory: {os.getcwd()}')
!git log --oneline -3


In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
!pip install -q ccxt pandas_ta 'ray[tune]>=2.5' pyarrow

# Verify key imports
import sys
sys.path.insert(0, 'src')

from adan_trading_bot.common.config_loader import ConfigLoader
from adan_trading_bot.environment.multi_asset_chunked_env import MultiAssetChunkedEnv
print('All imports successful!')


## 2. Data - Download 50,000 Real Candles via CCXT

Downloads BTC/USDT candles from public exchanges (no API key needed).
The script tries Binance > Bybit > Bitget automatically.


In [ ]:
# Download 50,000 candles for BTCUSDT (all timeframes)
!python scripts/generate_colab_dataset.py \
    --symbols BTCUSDT \
    --candles 50000 \
    --split train

# Verify the data
import pandas as pd
df = pd.read_parquet('data/processed/indicators/train/BTCUSDT/5m.parquet')
print(f'\nBTCUSDT 5m: {len(df)} rows, {len(df.columns)} columns')
print(f'Date range: {df.index.min()} to {df.index.max()}')
print(f'Columns: {list(df.columns)}')


## 3. Train - PBT Training with 4 Worker Profiles on T4 GPU

The training script auto-detects Colab and configures:
- `num_cpus=2` (Colab limit)
- `device="cuda"` (T4 GPU)
- `resources_per_trial={cpu: 0.5, gpu: 0.25}` (4 workers share T4)

Worker profiles:
- **Scalper** (5m): gamma=0.95, n_steps=512
- **Intraday** (1h): gamma=0.99, n_steps=2048
- **Swing** (4h): gamma=0.995, n_steps=8192
- **Position** (4h): gamma=0.999, n_steps=16384


In [ ]:
# Train with 4 profiles on T4 GPU
# Adjust --steps as needed: 1M for full training, 100K for quick test
!python scripts/train_parallel_agents.py \
    --config config/config.yaml \
    --steps 1000000 \
    --profiles scalper intraday swing position \
    --num-samples 4 \
    --steps-per-iter 10000


In [ ]:
# Alternative: Quick training run (for testing the pipeline)
# Uncomment the line below for a quick 30K step test
# !python scripts/train_simple_ppo.py --steps 30000


## 4. Extract - Best Model to Production

Scans all Ray Tune results, picks the trial with the highest reward,
and copies model + normalizer to `models/rl_agents/production/`.


In [ ]:
# Extract the best model
!python scripts/extract_best_model.py --metric mean_reward

# Verify extraction
import os
prod_dir = 'models/rl_agents/production'
if os.path.exists(f'{prod_dir}/model.zip'):
    size_mb = os.path.getsize(f'{prod_dir}/model.zip') / 1e6
    print(f'\nProduction model: {prod_dir}/model.zip ({size_mb:.1f} MB)')
    if os.path.exists(f'{prod_dir}/extraction_metadata.json'):
        import json
        with open(f'{prod_dir}/extraction_metadata.json') as f:
            meta = json.load(f)
        print(f'Source: {meta.get("source_trial", "?")}')
        print(f'Reward: {meta.get("mean_reward", 0):.4f}')
        print(f'Sharpe: {meta.get("mean_sharpe", 0):.4f}')
else:
    print('No production model found. Check training output.')


## 5. Paper Trading - Isolated Virtual Wallet

**IMPORTANT:** The paper trading monitor uses a 100% isolated virtual wallet.
- Initial balance: $20.50
- Max balance: $25.00 (Micro Capital tier)
- NO real orders are ever placed
- Binance Testnet is used ONLY for candle data (read-only)

### Option A: Offline mode (no API keys needed)
Uses locally downloaded data to simulate trading.

### Option B: Live mode (requires Testnet API keys)
Set your Binance Testnet keys in Colab Secrets.


In [ ]:
# Option A: Offline paper trading (no API keys needed)
# Runs for 10 minutes using local data
!python scripts/paper_trading_monitor.py --offline --duration 10


In [ ]:
# Option B: Live paper trading with Binance Testnet
# First, set your keys in Colab Secrets (key icon in left sidebar)
# Name them: BINANCE_TESTNET_KEY and BINANCE_TESTNET_SECRET

# from google.colab import userdata
# import os
# os.environ['BINANCE_API_KEY'] = userdata.get('BINANCE_TESTNET_KEY')
# os.environ['BINANCE_SECRET_KEY'] = userdata.get('BINANCE_TESTNET_SECRET')
# os.environ['BINANCE_TESTNET'] = 'true'
# !python scripts/paper_trading_monitor.py --duration 60


## 6. Validation - Trade Lifecycle Check

Run the lifecycle validator to ensure all trading rules are enforced.


In [ ]:
# Run a quick training and validate lifecycle
!python scripts/train_simple_ppo.py --steps 1000 2>&1 | tee /tmp/quick_train.log
!python scripts/validate_trade_lifecycle.py /tmp/quick_train.log --run-id "Colab-Quick"


## 7. Results & Download

Download the trained model for local use.


In [ ]:
# Package production model for download
import shutil
prod_dir = 'models/rl_agents/production'
if os.path.exists(prod_dir):
    shutil.make_archive('/content/adan_production_model', 'zip', prod_dir)
    print('Model packaged: /content/adan_production_model.zip')
    print('Download from the Files panel (folder icon) in Colab.')
else:
    print('No production model to package.')

# Show paper trading report if available
import json
report_path = 'results/paper_trading_report.json'
if os.path.exists(report_path):
    with open(report_path) as f:
        report = json.load(f)
    print(f'\nPaper Trading Results:')
    print(f'  Trades: {report.get("total_trades", 0)}')
    print(f'  Return: {report.get("total_return_pct", 0):+.2f}%')
    print(f'  Win Rate: {report.get("win_rate_pct", 0):.1f}%')
